In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/My Drive

/content/drive/My Drive


In [3]:
# Install required packages
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters
!pip install -q chromadb pypdf beautifulsoup4 sentence-transformers faiss-cpu
!pip install -q openai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [4]:
# Import libraries
import os
import numpy as np
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

# Set OpenAI API key
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
HF_API_KEY = userdata.get('HUGGINGFACE_API_KEY')
os.environ["HUGGINGFACE_API_KEY"] = HF_API_KEY
serper_api_key = userdata.get('SERPER_API_KEY')
os.environ["SERPER_API_KEY"] = serper_api_key
serp_api_key = userdata.get('SERPER_API_KEY')
os.environ["SERPER_API_KEY"] = serp_api_key
print("✓ Setup complete!")

✓ Setup complete!


In [5]:
from langchain_community.document_loaders import CSVLoader

# Import medreason instruction dataset
loader = CSVLoader('medreason-instruction-dataset.csv')
docs = loader.load()

print(f"Loaded {len(docs)} document(s)")
print(f"\nTotal characters: {len(docs[0].page_content):,}")
print(f"\nFirst 500 characters:\n{docs[0].page_content[:500]}...")
print(f"\nMetadata: {docs[0].metadata}")

Loaded 31535 document(s)

Total characters: 67

First 500 characters:
query: Most sensitive test for H pylori
answer: D. Urea breath test...

Metadata: {'source': 'medreason-instruction-dataset.csv', 'row': 0}


Split documents into smaller chunks for embedding and retrieval.
RecursiveCharacterTextSplitter

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,            # Maximum chunk size (characters), reduced for more granularity
    chunk_overlap=100,          # Overlap between chunks (20% of chunk_size)
    add_start_index=True,       # Track position in original document
    separators=["\n\n", "\n", " ", ""]  # Try separators in order
)

# Split the loaded documents
all_splits = text_splitter.split_documents(docs)

print(f"Split each document into {len(all_splits)} chunks")
print(f"\nChunk 0 length: {len(all_splits[0].page_content)} characters")
print(f"Chunk 0 metadata: {all_splits[0].metadata}")
print(f"\nFirst chunk content:\n{all_splits[0].page_content}")

Split each document into 35056 chunks

Chunk 0 length: 67 characters
Chunk 0 metadata: {'source': 'medreason-instruction-dataset.csv', 'row': 0, 'start_index': 0}

First chunk content:
query: Most sensitive test for H pylori
answer: D. Urea breath test


Check Chunk Overlap

In [7]:
# Demonstrate overlap with simple example
simple_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)

sample_text = """Artificial intelligence is transforming how we interact with technology.
Machine learning models can now understand and generate human language with remarkable accuracy.
This has enabled new applications in search, customer service, and content creation."""

simple_chunks = simple_splitter.split_text(sample_text)

print(f"Created {len(simple_chunks)} chunks with overlap:\n")
for i, chunk in enumerate(simple_chunks):
    print(f"Chunk {i+1} ({len(chunk)} chars): {chunk}")
    print(f"{'-'*80}")

Created 7 chunks with overlap:

Chunk 1 (46 chars): Artificial intelligence is transforming how we
--------------------------------------------------------------------------------
Chunk 2 (32 chars): how we interact with technology.
--------------------------------------------------------------------------------
Chunk 3 (46 chars): Machine learning models can now understand and
--------------------------------------------------------------------------------
Chunk 4 (43 chars): and generate human language with remarkable
--------------------------------------------------------------------------------
Chunk 5 (9 chars): accuracy.
--------------------------------------------------------------------------------
Chunk 6 (44 chars): This has enabled new applications in search,
--------------------------------------------------------------------------------
Chunk 7 (47 chars): search, customer service, and content creation.
---------------------------------------------------------------------

Compare Different Splitters

In [8]:
from langchain_text_splitters import CharacterTextSplitter

# CharacterTextSplitter (splits only on specified separator)
char_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separator="\n\n"  # Only split on double newlines
)

char_splits = char_splitter.split_documents(docs)

print(f"RecursiveCharacterTextSplitter: {len(all_splits)} chunks")
print(f"CharacterTextSplitter: {len(char_splits)} chunks")
print(f"\nRecursive splitter creates more uniform chunks by trying multiple separators.")

RecursiveCharacterTextSplitter: 35056 chunks
CharacterTextSplitter: 31758 chunks

Recursive splitter creates more uniform chunks by trying multiple separators.


Create embeddings and store them in a vector database.Create a vector store with Chroma.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings # Import HuggingFace embeddings

# Initialize embedding model
# Original: embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
# Change to HuggingFace embeddings for better retrieval for this dataset
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create vector store from document chunks
persist_directory = './chroma_db_hf_rerun' # Using a new directory to avoid conflicts and re-embed

vectordb = Chroma.from_documents(
    documents=all_splits,
    embedding=embedding_model,
    persist_directory=persist_directory
)

print(f"✓ Vector store created with {vectordb._collection.count()} document chunks using HuggingFace embeddings")
print(f"✓ Persisted to: {persist_directory}")

/tmp/ipykernel_128344/1813991882.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Load existing vector store.

In [9]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Initialize embedding model with the same model used for creation (sentence-transformers/all-MiniLM-L6-v2)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Set the persist directory to the existing HuggingFace database path
persist_directory = './chroma_db_hf_rerun'

# Load existing vector store (no need to re-embed)
vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_model
)

print(f"✓ Loaded vector store with {vectordb._collection.count()} documents using HuggingFace embeddings")

/tmp/ipykernel_627/111787739.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_627/111787739.py:11: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(


✓ Loaded vector store with 142677 documents using HuggingFace embeddings


Retrieval: Basic similarity search

Similarity search with scores

In [10]:
# Get documents with similarity scores
docs_with_scores = vectordb.similarity_search_with_score(question, k=3)

print(f"Query: {question}\n")

for i, (doc, score) in enumerate(docs_with_scores):
    print(f"Document {i+1} - Similarity Score: {score:.4f}")
    print(f"Content preview: {doc.page_content[:200]}...")
    print(f"{'-'*80}\n")

NameError: name 'question' is not defined

Maximum Marginal Relevance (MMR)

In [11]:
question = 'What does CPK measure?'

# Simulate history for testing purposes
history = []
model_choice = list(llama_models.keys())[0] # Select the first available Llama model

updated_history = generate_chat(question, history, model_choice)

print(f"\nQuestion: {question}")
print(f"AI Response: {updated_history[-1]['content']}")

NameError: name 'llama_models' is not defined

Metadata Filtering: Filter results based on metadata.

In [ ]:
# First, let's see what metadata is available
sample_doc = all_splits[0]
print("Available metadata fields:")
for key, value in sample_doc.metadata.items():
    print(f"  {key}: {value}")

# Example: Filter by source (if you have multiple sources)
# Note: This example uses the web source we loaded
source_filter = sample_doc.metadata.get('source')

print(f"\nFiltering by source: {source_filter}")

docs_filtered = vectordb.similarity_search(
    "Which of the following scoring system is used to see chest involvement in Sarcoidosis?",
    k=3,
    filter={"source": source_filter}
)

print(f"\nRetrieved {len(docs_filtered)} filtered documents")
for i, doc in enumerate(docs_filtered):
    print(f"Doc {i+1} source: {doc.metadata.get('source')}")

Available metadata fields:
  source: medreason-instruction-dataset.csv
  row: 0
  start_index: 0

Filtering by source: medreason-instruction-dataset.csv

Retrieved 3 filtered documents
Doc 1 source: medreason-instruction-dataset.csv
Doc 2 source: medreason-instruction-dataset.csv
Doc 3 source: medreason-instruction-dataset.csv


Question Answering with RAG

Combine retrieval with LLM generation.

RAG with Langchain agent

In [12]:
from langchain.agents import create_agent
from langchain.tools import tool

# Create retrieval tool
@tool
def retrieve_context(query: str) -> str:
    """Retrieve information from the knowledge base to help answer questions."""
    print(f"Retrieving context for query: {query}")
    retrieved_docs = vectordb.similarity_search(query, k=3)
    serialized = "\n\n".join(
        f"Source: {doc.metadata}\nContent: {doc.page_content}"
        for doc in retrieved_docs
    )
    return serialized

# Create agent with tools
agent = create_agent(
    #model="gpt-4.1-nano",
    model="gpt-4.1-nano",
    tools=[retrieve_context],
    #temperature=0,
    system_prompt="You are a helpful AI assistant. Use the retrieve_context tool to find information from the knowledge base when needed to answer questions accurately."
)

print("✓ RAG Agent created successfully!")

✓ RAG Agent created successfully!


Interactive RAG & QA

Evaluation: Retrieval Quality Metrics

In [14]:
# Create a simple test set
# In practice, you'd have ground truth labels

def evaluate_retrieval(query, k=5):
    """
    Evaluate retrieval quality.
    Note: This is a simplified version. In practice, you'd have ground truth.
    """
    # Retrieve documents
    docs = vectordb.similarity_search_with_score(query, k=k)

    print(f"Query: {query}")
    print(f"Retrieved {len(docs)} documents:\n")

    for i, (doc, score) in enumerate(docs):
        print(f"Rank {i+1} - Score: {score:.4f}")
        print(f"Content: {doc.page_content[:100]}...")
        print(f"{'-'*60}\n")

    # Calculate score statistics
    scores = [score for _, score in docs]
    print(f"\nScore Statistics:")
    print(f"  Mean: {np.mean(scores):.4f}")
    print(f"  Std:  {np.std(scores):.4f}")
    print(f"  Min:  {np.min(scores):.4f}")
    print(f"  Max:  {np.max(scores):.4f}")

# Test retrieval quality
evaluate_retrieval("Which flexor muscle is attached to hook of hamate?")

Query: Which flexor muscle is attached to hook of hamate?
Retrieved 5 documents:

Rank 1 - Score: 0.1609
Content: query: Which flexor muscle is attached to hook of hamate?
answer: C. Flexor digiti minimi...
------------------------------------------------------------

Rank 2 - Score: 0.1609
Content: query: Which flexor muscle is attached to hook of hamate?
answer: C. Flexor digiti minimi...
------------------------------------------------------------

Rank 3 - Score: 0.1609
Content: query: Which flexor muscle is attached to hook of hamate?
answer: C. Flexor digiti minimi...
------------------------------------------------------------

Rank 4 - Score: 0.1609
Content: query: Which flexor muscle is attached to hook of hamate?
answer: C. Flexor digiti minimi...
------------------------------------------------------------

Rank 5 - Score: 0.1609
Content: query: Which flexor muscle is attached to hook of hamate?
answer: C. Flexor digiti minimi...
---------------------------------------------

Compare Retrieval Strategies

In [15]:
def compare_retrieval_methods(query, k=3):
    """
    Compare different retrieval strategies.
    """
    print(f"Query: {query}\n")

    # Method 1: Similarity Search
    print("="*80)
    print("METHOD 1: Similarity Search")
    print("="*80)
    docs_sim = vectordb.similarity_search(query, k=k)
    for i, doc in enumerate(docs_sim):
        print(f"\n[{i+1}] {doc.page_content[:150]}...")

    # Method 2: MMR
    print("\n" + "="*80)
    print("METHOD 2: Maximum Marginal Relevance (MMR)")
    print("="*80)
    docs_mmr = vectordb.max_marginal_relevance_search(query, k=k, fetch_k=20)
    for i, doc in enumerate(docs_mmr):
        print(f"\n[{i+1}] {doc.page_content[:150]}...")

    # Check overlap
    overlap = sum(1 for d1 in docs_sim if any(
        d1.page_content == d2.page_content for d2 in docs_mmr
    ))

    print(f"\n{'='*80}")
    print(f"Overlap: {overlap}/{k} documents are the same")
    print(f"MMR provides {'more' if overlap < k else 'similar'} diversity")

compare_retrieval_methods("Which lesion displays an ill-defined border?")

Query: Which lesion displays an ill-defined border?

METHOD 1: Similarity Search

[1] query: Which lesion displays an ill-defined border?
answer: B. Sclerosing osteitis...

[2] query: Which lesion displays an ill-defined border?
answer: B. Sclerosing osteitis...

[3] query: Which lesion displays an ill-defined border?
answer: B. Sclerosing osteitis...

METHOD 2: Maximum Marginal Relevance (MMR)

[1] query: Which lesion displays an ill-defined border?
answer: B. Sclerosing osteitis...

[2] answer: Different pigmentation throughout the lesion...

[3] the patient’s genital lesions contain clear fluid and measure 5–6 mm in diameter. What is the appropriate description of these lesions?...

Overlap: 3/3 documents are the same
MMR provides similar diversity


Answer Quality Assessment

In [16]:
# Simple answer quality check using LLM
from langchain_core.prompts import PromptTemplate

eval_template = """Evaluate the following answer based on the provided context.

Context: {context}

Question: {question}

Answer: {answer}

Evaluation Criteria:
1. Faithfulness: Is the answer supported by the context? (Yes/No)
2. Relevance: Does the answer address the question? (Yes/No)
3. Completeness: Is the answer complete? (Yes/No)

Provide your evaluation:"""

eval_prompt = PromptTemplate.from_template(eval_template)
eval_chain = eval_prompt | llm | StrOutputParser()

def evaluate_answer(question):
    # Generate answer
    answer, docs = rag_qa(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    print(f"Question: {question}")
    print(f"\nAnswer: {answer}")

    # Evaluate
    evaluation = eval_chain.invoke({
        "context": context,
        "question": question,
        "answer": answer
    })

    print(f"\n{'='*80}")
    print("EVALUATION:")
    print(f"{'='*80}")
    print(evaluation)

evaluate_answer("Which lesion displays an ill-defined border?")

NameError: name 'llm' is not defined

Trying different chuck sizes

In [ ]:
# Test different chunk sizes
chunk_sizes = [500, 1000, 1500]

print("Testing different chunk sizes:\n")

for size in chunk_sizes:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=int(size * 0.2),  # 20% overlap
        add_start_index=True
    )

    splits = splitter.split_documents(docs)

    print(f"Chunk size: {size}")
    print(f"  Total chunks: {len(splits)}")
    print(f"  Avg chunk length: {np.mean([len(s.page_content) for s in splits]):.1f}")
    print(f"  Min chunk length: {min([len(s.page_content) for s in splits])}")
    print(f"  Max chunk length: {max([len(s.page_content) for s in splits])}")
    print()

print("💡 Smaller chunks → more granular retrieval but may lose context")
print("💡 Larger chunks → more context but less precise retrieval")

Testing different chunk sizes:

Chunk size: 500
  Total chunks: 35056
  Avg chunk length: 194.6
  Min chunk length: 3
  Max chunk length: 500

Chunk size: 1000
  Total chunks: 32048
  Avg chunk length: 209.7
  Min chunk length: 9
  Max chunk length: 1000

Chunk size: 1500
  Total chunks: 31669
  Avg chunk length: 211.7
  Min chunk length: 9
  Max chunk length: 1500

💡 Smaller chunks → more granular retrieval but may lose context
💡 Larger chunks → more context but less precise retrieval


Alternative Embedding Models: HuggingFace Embeddings

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Use a free open-source embedding model
hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Test embeddings
test_text = "This is a test sentence."
hf_embedding = hf_embeddings.embed_query(test_text)

print(f"HuggingFace Embedding dimension: {len(hf_embedding)}")
print(f"First 10 values: {hf_embedding[:10]}")

# Create vector store with HuggingFace embeddings
vectordb_hf = Chroma.from_documents(
    documents=all_splits[:50],  # Use subset for faster demo
    embedding=hf_embeddings,
    persist_directory='./chroma_db_hf'
)

print(f"\n✓ Created vector store with HuggingFace embeddings")
print(f"✓ Contains {vectordb_hf._collection.count()} documents")

/tmp/ipykernel_5971/1754853651.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  hf_embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HuggingFace Embedding dimension: 384
First 10 values: [0.08429647237062454, 0.057953689247369766, 0.004493385087698698, 0.10582111030817032, 0.007083410397171974, -0.017844678834080696, -0.016888074576854706, -0.015228300355374813, 0.0404730923473835, 0.033422548323869705]

✓ Created vector store with HuggingFace embeddings
✓ Contains 250 documents


Log in to Hugging Face Hub

In [17]:
from huggingface_hub import login
login()

Import Required Libraries

In [18]:
pip install gradio

In [19]:
import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [20]:
device = 0 if torch.cuda.is_available() else -1

In [21]:
llama_models = {
    #"Llama 3 70B Instruct": "meta-llama/Meta-Llama-3-70B-Instruct",
    #"Llama 3 8B Instruct": "meta-llama/Meta-Llama-3-8B-Instruct",
    #"Llama 3.1 70B Instruct": "meta-llama/Llama-3.1-70B-Instruct",
    #"Llama 3.1 8B Instruct": "meta-llama/Llama-3.1-8B-Instruct",
    "Llama 3.2 3B Instruct": "llama3", # Changed to 'llama3' as pulled by Ollama
    #"Llama 3.2 1B Instruct": "meta-llama/Llama-3.2-1B-Instruct",
}

Loading Models

In [ ]:
def load_model(model_name):
    """Load the specified model."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    generator = pipeline('text-generation', model=model, tokenizer=tokenizer, device=device)
    return generator

In [22]:
# Cahce Model
model_cache = {}

Chat Generation

In [23]:
import sys

def generate_chat(user_input, history, model_choice):
    """Generate q&a assistant responses using the selected model and task."""

    system_prompt = {"role": "system", "content": "You are a helpful assistant"}

    if history is None:
        history = [system_prompt]

    history.append({"role": "user", "content": user_input})

    ollama_model_id = llama_models.get(model_choice, "llama3") # Default to llama3 if not found

    question = user_input

    # Ensure vectordb and embedding_model are loaded/accessible within the Gradio context
    global vectordb
    global embedding_model

    if vectordb is None or not hasattr(vectordb, '_collection') or vectordb._collection.count() == 0:
        print("--- DEBUGGING: vectordb not loaded or empty in generate_chat, attempting reload ---")
        sys.stdout.flush()
        if embedding_model is None:
            from langchain_community.embeddings import HuggingFaceEmbeddings
            embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        try:
            vectordb = Chroma(
                persist_directory='./chroma_db_hf_rerun',
                embedding_function=embedding_model
            )
            print(f"--- DEBUGGING: vectordb reloaded with {vectordb._collection.count()} documents ---")
            sys.stdout.flush()
        except Exception as e:
            error_msg = f"Error: Knowledge base not available. Failed to load vector store: {e}"
            print(f"--- DEBUGGING: {error_msg} ---")
            sys.stdout.flush()
            history.append({"role": "assistant", "content": error_msg})
            return history # Return history with error message

    # --- DEBUGGING vectordb state: Start ---
    if vectordb is None:
        print("--- DEBUGGING: vectordb is None ---")
        sys.stdout.flush()
    else:
        print(f"--- DEBUGGING: vectordb contains {vectordb._collection.count()} documents ---")
        sys.stdout.flush()
        # Removed: print(f"--- DEBUGGING: vectordb persist_directory: {vectordb.persist_directory} ---")
    # --- DEBUGGING vectordb state: End ---

    # Retrieve top-k similar documents from the ChromaDB vector store using MMR
    docs_retrieved = vectordb.max_marginal_relevance_search(question, k=1, fetch_k=20) # Changed k back to 1 for precise answer

    # --- DEBUGGING OUTPUT (RETRIEVAL): Start ---
    print(f"\n--- DEBUGGING (RETRIEVAL): Retrieved {len(docs_retrieved)} documents for query: '{question}' ---")
    sys.stdout.flush()
    if len(docs_retrieved) == 0:
        print("No documents retrieved from vector store.")
        sys.stdout.flush()
    for i, doc in enumerate(docs_retrieved):
        print(f"Document {i+1} (Source: {doc.metadata.get('source', 'Unknown')}, Start Index: {doc.metadata.get('start_index', 'N/A')}):\nContent: {doc.page_content[:500]}...\n---------------------------------------------------") # Print more content for debugging
        sys.stdout.flush()
    print("--- DEBUGGING (RETRIEVAL): End ---")
    sys.stdout.flush()

    base_instruction_prompt = """
    Answer the question based on your medical knowledge derived from medical websites such as www.webmd.com, www.clevelandclinic.org, www.mayoclinic.org, www.medlineplus.com and medical blogs only. Start your answer with 'According to medical websites, including MedlinePlus, Cleveland Clinic and Mayo Clinic'. Keep the answer short and concise. Respond "I cannot find the best answer. Please consult with a Doctor." if not sure about the answer.
    If the question is not related to the context, politely respond that 'I Apologize. I am only programmed to answer medical questions.' and do not fetch youtube results

    """

    # Prompt specifically for out-of-context questions, without attribution
    out_of_context_prompt_text = "I Apologize. I am only programmed to answer medical questions."

    llm_answer = ""

    if len(docs_retrieved) == 0:
      print(f"No relevant documents found for query: {question}")
      sys.stdout.flush()
      # Construct a general prompt for the LLM when no context is available
      final_prompt = f"""
      {base_instruction_prompt}
      Question: {question}

      Answer:
      """
      llm_answer = get_completion(final_prompt, model_name=ollama_model_id)
    else:
      extracted_answers = []
      for doc in docs_retrieved:
          content = doc.page_content
          # Assuming the format is always "query: ...\nanswer: ..."
          if "answer:" in content:
              answer_part = content.split("answer:", 1)[1].strip()
              extracted_answers.append(answer_part)
          else:
              # Fallback if the format is not as expected, use full content
              extracted_answers.append(content)

      # Format extracted answers as context for the RAG prompt
      context = "\\n\\n".join(extracted_answers)

      # Construct the RAG prompt including the base instructions and retrieved context
      final_prompt = f"""
      Use the following pieces of context to answer the question at the end. Start your answer with 'According to medical websites, including MedlinePlus, Cleveland Clinic and Mayo Clinic'.
      If you don't know the answer, just say that you don't know, don't try to make up an answer.
      If the question is not related to the context, politely respond 'I Apologize. I am only programmed to answer medical questions.' and do not fetch youtube results
      Use three sentences maximum. Keep the answer as concise as possible.

      Context: {context}

      Question: {question}

      Helpful Answer:"""

      # Call get_completion with the constructed RAG prompt
      llm_answer = get_completion(final_prompt, model_name=ollama_model_id)

      print(f"Query: {question}")
      sys.stdout.flush()
      print(f"\\nRetrieved {len(docs_retrieved)} documents for context:")
      sys.stdout.flush()
      for i, doc in enumerate(docs_retrieved):
          print(f"{"=" * 80}")
          sys.stdout.flush()
          print(f"Document {i+1}")
          sys.stdout.flush()
          print(f"{"=" * 80}")
          sys.stdout.flush()
          print(f"Source: {doc.metadata.get('source', 'Unknown')}")
          sys.stdout.flush()
          print(f"Start Index: {doc.metadata.get('start_index', 'N/A')}")
          sys.stdout.flush()
          print(f"\\nContent:\\n{doc.page_content[:300]}...\\n")
          sys.stdout.flush()

    # --- DEBUGGING: llm_answer and condition check ---
    print(f"--- DEBUGGING: llm_answer content: {repr(llm_answer)} ---")
    sys.stdout.flush()
    check_phrase = out_of_context_prompt_text.lower().strip() # Use the exact phrase for checking
    is_out_of_context = check_phrase in llm_answer.lower().strip()
    print(f"--- DEBUGGING: Is out of context phrase found? {is_out_of_context} ---")
    sys.stdout.flush()
    # --- DEBUGGING: End ---

    # Check if LLM explicitly states it's out of context
    if is_out_of_context:
        response_content = out_of_context_prompt_text # Return only the apology, no YouTube search
    else:
        # Always perform YouTube search if question is in context or fallback is triggered
        youtube_query = f"best medical explanation review {question} MedlinePlus Cleveland Clinic Mayo Clinic,3"
        try:
            raw_youtube_results = youtube_tool.run(youtube_query)
            # Parse and format YouTube results into a numbered list
            import ast
            try:
                youtube_links = ast.literal_eval(raw_youtube_results)
                if isinstance(youtube_links, list):
                    formatted_youtube_results = "\n".join([f"{i+1}. {link}" for i, link in enumerate(youtube_links)])
                else:
                    formatted_youtube_results = raw_youtube_results # Fallback if parsing fails
            except (ValueError, SyntaxError):
                formatted_youtube_results = raw_youtube_results # Fallback if not a parsable list string
        except Exception as e:
            formatted_youtube_results = f"Error fetching YouTube results: {e}"
            print(f"--- DEBUGGING: YouTube tool error: {e} ---")
            sys.stdout.flush()

        # Combine the LLM's answer with YouTube results
        response_content = f"{llm_answer}\n\nAdditional relevant videos from YouTube:\n{formatted_youtube_results}"

    history.append({"role": "assistant", "content": response_content})

    return history

In [24]:
import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

with gr.Blocks(theme=gr.themes.Soft()) as demo: # Apply a theme for a polished look
    gr.Markdown(
        """
        <style>
            .gradio-container {
                border: 2px solid #a8dadc; /* Light blue/teal border */
                border-radius: 10px; /* Slightly rounded corners */
                box-shadow: 0 4px 8px rgba(0, 0, 0, 0.1); /* Subtle shadow */
                padding: 15px; /* Add some padding inside the border */
            }
            /* Target the chat window specifically */
            .gr-chatbot {
                border: 1px solid #d1e2ec; /* Lighter border for chatbox itself */
                border-radius: 8px;
                border-top: 3px solid #6c757d; /* Stronger top border for separation */
            }
        </style>
        <h1 style='text-align: center; margin-bottom: 1em;'>
            <img src='https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/Red_cross_icon_%28vector%29.svg/1200px-Red_cross_icon_%28vector%29.svg.png' alt='' style='height: 40px; vertical-align: middle; margin-right: 10px;'>
            MedIntel Q&A Assistant
        </h1>
        <p style='text-align: center; font-size: 1.1em; color: #555;'>
            Ask medical questions and get answers with knowledge base retrieval, LLM generation, and YouTube references.
        </p>
        """
    )

    with gr.Row():
        model_choice = gr.Dropdown(list(llama_models.keys()), label="Select a Llama Model for AI Generation", scale=1)

    with gr.Column(scale=4):
        chatbot = gr.Chatbot(label="Conversation History", height=400)
        txt_input = gr.Textbox(show_label=False, placeholder="Type your medical question here...", lines=2)

        with gr.Row():
            submit_btn = gr.Button("Submit Question", variant="primary", scale=1)
            clear_btn = gr.ClearButton(value="Clear Chat", scale=0)

    def respond(user_input, chat_history, model_choice):
        try:
            if not user_input:
                return "", chat_history # Don't process empty input

            if model_choice is None:
                model_choice = list(llama_models.keys())[0]

            # Ensure chat_history is a list of lists for Gradio Chatbot format
            if chat_history is None:
                chat_history = []

            # Gradio chatbot expects [[user_msg, ai_msg], ...]
            # Our generate_chat uses [{'role': 'user', 'content': '...'}, {'role': 'assistant', 'content': '...'}]
            # We need to convert from Gradio's chat_history format to generate_chat's history format
            converted_history = []
            for human, ai in chat_history:
                if human: converted_history.append({'role': 'user', 'content': human})
                if ai: converted_history.append({'role': 'assistant', 'content': ai})

            updated_generate_chat_history = generate_chat(user_input, converted_history, model_choice)

            # Convert generate_chat's history format back to Gradio's chat_history format
            new_chat_history = []
            for i in range(0, len(updated_generate_chat_history), 2):
                user_msg = updated_generate_chat_history[i]['content'] if i < len(updated_generate_chat_history) else None
                ai_msg = updated_generate_chat_history[i+1]['content'] if i+1 < len(updated_generate_chat_history) else None
                new_chat_history.append([user_msg, ai_msg])

            return "", new_chat_history
        except Exception as e:
            error_message = f"An error occurred: {e}"
            print(f"Error in respond function: {e}") # Print to Colab console

            if chat_history is None:
                chat_history = []
            chat_history.append([user_input, error_message]) # Append error to chat for user
            return "", chat_history

    submit_btn.click(respond, [txt_input, chatbot, model_choice], [txt_input, chatbot])
    txt_input.submit(respond, [txt_input, chatbot, model_choice], [txt_input, chatbot]) # Allow submit on Enter key
    clear_btn.click(lambda: None, None, chatbot, queue=False) # Clear chatbot on button click


/tmp/ipykernel_627/1552750033.py:5: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo: # Apply a theme for a polished look
/tmp/ipykernel_627/1552750033.py:36: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Conversation History", height=400)
/tmp/ipykernel_627/1552750033.py:36: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Conversation History", height=400)


In [25]:
!pip install python-dotenv

In [26]:
pip install ollama

Ollama Server

In [27]:
import subprocess

# Download and execute the Ollama installation script
process = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True, text=True)
print(process.stdout)
print(process.stderr)

ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd

>>> Installing ollama to /usr/local



In [28]:
import subprocess

process = subprocess.run("sudo apt-get update && sudo apt-get install -y zstd", shell=True, capture_output=True, text=True)
print(process.stdout)
print(process.stderr)

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,442 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-

In [29]:
from langchain_community.tools import YouTubeSearchTool
youtube_tool = YouTubeSearchTool()

In [30]:
!pip install youtube_search

In [31]:
import subprocess

# Download and execute the Ollama installation script again
process = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True, text=True)
print(process.stdout)
print(process.stderr)


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
#=#=#                                                                         
##O#-#                                                                        
##O=#  #                                                                      

                                                                           0.0%
                                                                           0.1%
                                                                           0.2%
                                                                           0.5%
                                                                           0.7%
                                                                           0.8%
                                                                           0.9%
                                                                     

In [32]:
import subprocess

# Verify if Ollama is running by sending a request to its default API endpoint
process = subprocess.run("curl -s http://localhost:11434/api/tags", shell=True, capture_output=True, text=True)
print(process.stdout)
print(process.stderr)

In [35]:
import subprocess

# Pull the 'llama3' model using ollama pull command
process = subprocess.run("ollama pull llama3", shell=True, capture_output=True, text=True)
print(process.stdout)
print(process.stderr)


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling 6a0746a1ec1a:   0% ▕                  ▏  41 KB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   0% ▕                  ▏  13 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   1% ▕                  ▏  61 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   2% ▕                  ▏  77 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   2% ▕                  ▏ 114 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   3% ▕                  ▏ 137 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   4% ▕                  ▏ 171 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   4% ▕                  ▏ 191 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   5% ▕      

In [36]:
import subprocess
import time

# Start Ollama server in the background
# Using nohup and & to run it in the background and detach from the current process
ollama_start_command = "nohup /usr/local/bin/ollama serve > ollama_server.log 2>&1 &"
subprocess.run(ollama_start_command, shell=True, capture_output=True, text=True)

# Give Ollama a few seconds to start up
time.sleep(5) # Wait for 5 seconds

# Verify if Ollama is running by sending a request to its default API endpoint
process = subprocess.run("curl -s http://localhost:11434/api/tags", shell=True, capture_output=True, text=True)
print(process.stdout)
print(process.stderr)

{"models":[{"name":"llama3:latest","model":"llama3:latest","modified_at":"2026-03-18T01:36:25.824905566Z","size":4661224676,"digest":"365c0bd3c000a25d28ddbf732fe1c6add414de7275464c4e4d1c3b5fcb5d8ad1","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_size":"8.0B","quantization_level":"Q4_0"}}]}



In [37]:
from ollama import chat

def get_completion(prompt_string, model_name="llama3"):
    messages = [{"role": "user", "content": prompt_string}]
    response = chat(
        model=model_name,
        messages=messages,
        options={
            "temperature": 0, # Set to 0 for deterministic output during testing
            "top_p": 0,       # Set to 0 for deterministic output during testing
            "num_predict": 512, # Equivalent to max_length for Ollama
            "num_keep": 0, # To avoid truncating the start of the response
            "mirostat_tau": 5.0, # Mirostat sampling for controlling perplexity
            "mirostat_eta": 0.1, # Learning rate for Mirostat
            # Removed: "stop": ["\n"], # Stop generation at newline
        }
    )
    return response['message']['content']

Prompting

In [ ]:
prompt = f"""
Answer the question based on the context below. Start your answer with 'According to medical websites, including MedlinePlus and Cleveland Clinic,'. Keep the answer short and concise. Respond "I cannot find the best answer. Please consult with a Doctor." if not sure about the answer.

Context: Search for answers on medical websites such as www.webmd.com,www.clevelandclinic.org,www.medlineplus.com and medical blogs only.

Question: What does GOT measure?

Answer:

"""
response = get_completion(prompt)
print("Completion for Text:")
print(response)

Completion for Text:
According to medical websites, including MedlinePlus and Cleveland Clinic, Gamma-Glutamyl Transferase (GOT) is an enzyme that measures liver function and is often used as a marker for liver damage or disease.


In [ ]:
prompt = f"""
Answer the question based on the context below. Keep the answer short and concise. Respond "I cannot find the best answer. Please consult with a Doctor." if not sure about the answer.

Context: Search for answers on medical websites such as www.webmd.com,www.clevelandclinic.org,www.medlineplus.com and medical blogs only.

Question: What is the difference between flu and rotovirus?

Answer:

"""
response = get_completion(prompt)
print("Completion for Text:")
print(response)

Completion for Text:
According to reputable medical sources, including WebMD and MedlinePlus, the main difference between flu (influenza) and rotavirus is that influenza is a viral respiratory illness caused by the influenza virus, whereas rotavirus is a highly contagious virus that causes severe diarrhea and vomiting in children. Rotavirus is a common cause of gastroenteritis (stomach flu) in young children.


In [ ]:
prompt = f"""
Answer the question based on the context below. Keep the answer short and concise. Respond "I cannot find the best answer. Please consult with a Doctor." if not sure about the answer.

Context: Search for answers on medical websites such as www.webmd.com,www.clevelandclinic.org,www.medlineplus.com and medical blogs only.

Question: What should I do if I'm feeling pain in my chest?

Answer:

"""
response = get_completion(prompt)
print("Completion for Text:")
print(response)

Completion for Text:
If you're experiencing chest pain, call emergency services or visit the emergency room immediately. Chest pain can be a sign of a heart attack or other serious condition. Do not delay seeking medical attention.


In [ ]:
prompt = f"""
Answer the question based on the context below. Keep the answer short and concise. Respond "I cannot find the best answer. Please consult with a Doctor." if not sure about the answer.

Context: Search for answers on medical websites such as www.webmd.com,www.clevelandclinic.org,www.medlineplus.com and medical blogs only.

Question: What does TSH measure?

Answer:

"""
response = get_completion(prompt)
print("Completion for Text:")
print(response)

Completion for Text:
TSH (Thyroid-Stimulating Hormone) measures the level of thyroid hormones in your blood. It indicates how well your thyroid gland is functioning, specifically if it's overactive or underactive.


In [ ]:
prompt = f"""
Answer the question based on the context below. Keep the answer short and concise. Respond "I cannot find the best answer. Please consult with a Doctor." if not sure about the answer.

Context: Search for answers on medical websites such as www.webmd.com,www.clevelandclinic.org,www.medlineplus.com and medical blogs only.

Question: What does GPT measure?

Answer:

"""
response = get_completion(prompt)
print("Completion for Text:")
print(response)

Completion for Text:
GPT (Glucose Tolerance Test) measures blood sugar levels to diagnose diabetes or prediabetes.


In [ ]:
prompt = f"""
Answer the question based on the context below. Keep the answer short and concise. Respond "I cannot find the best answer. Please consult with a Doctor." if not sure about the answer.

Context: Search for answers on medical websites such as www.webmd.com,www.clevelandclinic.org,www.medlineplus.com and medical blogs only.

Question: What does FT4 measure?

Answer:

"""
response = get_completion(prompt)
print("Completion for Text:")
print(response)

Completion for Text:
FT4 measures Free Thyroxine (T4) levels in the blood, which is a hormone produced by the thyroid gland.


In [38]:
prompt = f"""
Answer the question based on the context below. Keep the answer short and concise. Respond "Unsure about answer" if not sure about the answer.

Question: ACTH is produced by which of the following Bronchogenic carcinomas?

Answer:

"""
response = get_completion(prompt)
print(response)

Small cell carcinoma.


In [ ]:
question = "What is the most common site of origin of thrombotic pulmonary emboli?"

# Retrieve top-k similar documents
docs_retrieved = vectordb.similarity_search(question, k=1)

print(f"Query: {question}")
print(f"\nRetrieved {len(docs_retrieved)} documents:\n")

for i, doc in enumerate(docs_retrieved):
    print(f"{'='*80}")
    print(f"Document {i+1}")
    print(f"{'='*80}")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")
    print(f"Start Index: {doc.metadata.get('start_index', 'N/A')}")
    print(f"\nContent:\n{doc.page_content[:300]}...\n")

Query: What is the most common site of origin of thrombotic pulmonary emboli?

Retrieved 1 documents:

Document 1
Source: medreason-instruction-dataset.csv
Start Index: 0

Content:
query: What is the most common site of origin of thrombotic pulmonary emboli?
answer: A. Deep leg veins...



In [ ]:
demo.close()

Launch the interface

In [39]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2edef363de7e535f83.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from langchain.tools import tool

@tool
def calculate_bmi(weight_kg: float, height_m: float) -> str:
    """Calculates Body Mass Index (BMI) given weight in kilograms and height in meters.
    Use this tool when the user asks for BMI and provides both weight and height.
    Example: calculate_bmi(weight_kg=70, height_m=1.75)
    """
    if height_m <= 0 or weight_kg <= 0:
        return "Error: Height and weight must be positive values."
    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5:
        category = "Underweight"
    elif 18.5 <= bmi < 24.9:
        category = "Normal weight"
    elif 25 <= bmi < 29.9:
        category = "Overweight"
    else:
        category = "Obesity"
    return f"Your BMI is {bmi:.2f}, which falls into the '{category}' category."

In [ ]:
from langchain.agents import create_agent
from langchain.agents.agent_executor import AgentExecutor # Corrected import path for AgentExecutor
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Initialize the LLM for the agent (using gpt-4.1-nano as per notebook's preference)
# Ensure OPENAI_API_KEY is set in your environment or Google Colab userdata
llm_agent = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

# Define the tools the agent can use
medical_tools = [calculate_bmi]

# Create the prompt for the agent
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful medical assistant. You have access to tools to help answer questions."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

# Create the agent using create_agent
agent = create_agent(llm_agent, medical_tools, agent_prompt)

# Create an agent executor to run the agent
agent_executor = AgentExecutor(agent=agent, tools=medical_tools, verbose=True)

ModuleNotFoundError: No module named 'langchain.agents.agent_executor'

In [ ]:
# Upgrade LangChain packages to ensure consistent imports
!pip install -q --upgrade langchain langchain-openai langchain-community

print("LangChain and related packages upgraded.")

LangChain and related packages upgraded.


In [ ]:
# Upgrade LangChain packages to ensure consistent imports
!pip install -q --upgrade langchain langchain-openai langchain-community

print("LangChain and related packages upgraded.")

LangChain and related packages upgraded.


In [ ]:
# Perform a clean reinstallation of LangChain packages
!pip uninstall -y langchain langchain-openai langchain-community
!pip install -q langchain langchain-openai langchain-community

print("LangChain and related packages reinstalled.")

Found existing installation: langchain 1.2.12
Uninstalling langchain-1.2.12:
  Successfully uninstalled langchain-1.2.12
Found existing installation: langchain-openai 1.1.11
Uninstalling langchain-openai-1.1.11:
  Successfully uninstalled langchain-openai-1.1.11
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
LangChain and related packages reinstalled.


In [ ]:
# Upgrade LangChain packages to ensure consistent imports
!pip install -q --upgrade langchain langchain-openai langchain-community

print("LangChain and related packages upgraded.")

LangChain and related packages upgraded.


In [ ]:
print("\n--- Testing BMI Agent ---")

try:
    response = agent_executor.invoke({"input": "I weigh 70 kilograms and my height is 1.75 meters. What is my BMI?"})
    print(f"\nAgent Response: {response['output']}")

    response_error = agent_executor.invoke({"input": "What is my BMI?"})
    print(f"\nAgent Response: {response_error['output']}")

    response_invalid = agent_executor.invoke({"input": "I weigh 60 kg and my height is 0 meters. What is my BMI?"})
    print(f"\nAgent Response: {response_invalid['output']}")

except Exception as e:
    print(f"Error running BMI agent: {e}")


--- Testing BMI Agent ---
Error running BMI agent: name 'agent_executor' is not defined


In [ ]:
app_code = """
import streamlit as sl
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

device = 0 if torch.cuda.is_available() else -1

llama_models = {
    "Llama 3.2 3B Instruct": "meta-llama/Llama-3.2-3B-Instruct",
}

def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    generator = pipeline('text-generation', model=model, tokenizer=tokenizer, device=device)
    return generator

model_cache = {}

def generate_chat(user_input, history, model_choice):
    if model_choice not in model_cache:
        model_cache[model_choice] = load_model(llama_models[model_choice])
    generator = model_cache[model_choice]

    system_prompt = {"role": "system", "content": "You are a helpful assistant"}

    if history is None:
        history = [system_prompt]

    history.append({"role": "user", "content": user_input})

    response = generator(
        history,
        max_length=512,
        pad_token_id=generator.tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )[-1]["generated_text"][-1]["content"]

    history.append({"role": "assistant", "content": response})

    return history

with sl.Blocks() as demo:
    sl.Markdown("<h1><center>Chat with Models</center></h1>")

    model_choice = sl.Dropdown(list(llama_models.keys()), label="Select Llama Model")

    chatbot = sl.Chatbot(label="Chatbot Interface", type = "messages")
    txt_input = sl.Textbox(show_label=False, placeholder="Type your message here...")

    def respond(user_input, chat_history, model_choice):
        if model_choice is None:
            model_choice = list(llama_models.keys())[0]
        updated_history = generate_chat(user_input, chat_history, model_choice)
        return "", updated_history

    txt_input.submit(respond, [txt_input, chatbot, model_choice], [txt_input, chatbot])

    submit_btn = sl.Button("Submit")
    submit_btn.click(respond, [txt_input, chatbot, model_choice], [txt_input, chatbot])

demo.launch()
"""

with open('app1.py', 'w') as f:
    f.write(app_code)
